In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, cross_validate

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

In [2]:
df = pd.read_csv('heart.csv')

Preprocess

In [3]:
categorical_data =df.select_dtypes(include='object')
pd.get_dummies(categorical_data)
df = pd.concat([df,pd.get_dummies(categorical_data)],axis=1)
df

C:\Users\cocoh\AppData\Local\Temp\ipykernel_4676\2939392443.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_data =df.select_dtypes(include='object')


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,...,ChestPainType_NAP,ChestPainType_TA,RestingECG_LVH,RestingECG_Normal,RestingECG_ST,ExerciseAngina_N,ExerciseAngina_Y,ST_Slope_Down,ST_Slope_Flat,ST_Slope_Up
0,40,M,ATA,140,289,0,Normal,172,N,0.0,...,False,False,False,True,False,True,False,False,False,True
1,49,F,NAP,160,180,0,Normal,156,N,1.0,...,True,False,False,True,False,True,False,False,True,False
2,37,M,ATA,130,283,0,ST,98,N,0.0,...,False,False,False,False,True,True,False,False,False,True
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,...,False,False,False,True,False,False,True,False,True,False
4,54,M,NAP,150,195,0,Normal,122,N,0.0,...,True,False,False,True,False,True,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,M,TA,110,264,0,Normal,132,N,1.2,...,False,True,False,True,False,True,False,False,True,False
914,68,M,ASY,144,193,1,Normal,141,N,3.4,...,False,False,False,True,False,True,False,False,True,False
915,57,M,ASY,130,131,0,Normal,115,Y,1.2,...,False,False,False,True,False,False,True,False,True,False
916,57,F,ATA,130,236,0,LVH,174,N,0.0,...,False,False,True,False,False,True,False,False,True,False


In [4]:
df = pd.get_dummies(df, dtype=int)
df

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease,Sex_F,Sex_M,ChestPainType_ASY,...,ChestPainType_NAP,ChestPainType_TA,RestingECG_LVH,RestingECG_Normal,RestingECG_ST,ExerciseAngina_N,ExerciseAngina_Y,ST_Slope_Down,ST_Slope_Flat,ST_Slope_Up
0,40,140,289,0,172,0.0,0,False,True,False,...,0,0,0,1,0,1,0,0,0,1
1,49,160,180,0,156,1.0,1,True,False,False,...,1,0,0,1,0,1,0,0,1,0
2,37,130,283,0,98,0.0,0,False,True,False,...,0,0,0,0,1,1,0,0,0,1
3,48,138,214,0,108,1.5,1,True,False,True,...,0,0,0,1,0,0,1,0,1,0
4,54,150,195,0,122,0.0,0,False,True,False,...,1,0,0,1,0,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,110,264,0,132,1.2,1,False,True,False,...,0,1,0,1,0,1,0,0,1,0
914,68,144,193,1,141,3.4,1,False,True,True,...,0,0,0,1,0,1,0,0,1,0
915,57,130,131,0,115,1.2,1,False,True,True,...,0,0,0,1,0,0,1,0,1,0
916,57,130,236,0,174,0.0,1,True,False,False,...,0,0,1,0,0,1,0,0,1,0


In [5]:
# Calculate the median from the NON-ZERO values
non_zero_bp = df[df['RestingBP'] > 0]['RestingBP']
bp_median = non_zero_bp.median()
print(f"Median RestingBP (non-zero values): {bp_median}") 

# Replace 0s with the calculated median
df['RestingBP'] = df['RestingBP'].replace(0, bp_median)

# Verify the fix- Check for zeros and new statistics
print(f"Number of zeros in RestingBP after replacement: {(df['RestingBP'] == 0).sum()}")
print("\nNew Summary Statistics for RestingBP:")
print(df['RestingBP'].describe())

Median RestingBP (non-zero values): 130.0
Number of zeros in RestingBP after replacement: 0

New Summary Statistics for RestingBP:
count    918.000000
mean     132.538126
std       17.990127
min       80.000000
25%      120.000000
50%      130.000000
75%      140.000000
max      200.000000
Name: RestingBP, dtype: float64


In [6]:
df.drop(df[df['RestingBP'] == 0].index, inplace=True)

In [7]:
# Calculate the median from the NON-ZERO values
non_zero_chl = df[df['Cholesterol'] > 0]['Cholesterol']
chl_median = non_zero_chl.median()
print(f"Median Cholesterol (non-zero values): {chl_median}") 

# Replace 0s with the calculated median
df['Cholesterol'] = df['Cholesterol'].replace(0, chl_median)

# Verify the fix- Check for zeros and new statistics
print(f"Number of zeros in Cholesterol after replacement: {(df['Cholesterol'] == 0).sum()}")
print("\nNew Summary Statistics for Cholesterol:")
print(df['Cholesterol'].describe())

Median Cholesterol (non-zero values): 237.0
Number of zeros in Cholesterol after replacement: 0

New Summary Statistics for Cholesterol:
count    918.000000
mean     243.204793
std       53.401297
min       85.000000
25%      214.000000
50%      237.000000
75%      267.000000
max      603.000000
Name: Cholesterol, dtype: float64


In [13]:
X = df.drop(columns = 'HeartDisease')
y = df['HeartDisease']

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print("Training set: ", X_train.shape, y_train.shape)
print("Test set: ", X_test.shape, y_test.shape)

Training set:  (734, 34) (734,)
Test set:  (184, 34) (184,)


Cross-Validation

In [17]:
duplicate_columns = X_train.columns[X_train.columns.duplicated()]

print("Duplicate columns:")
print(duplicate_columns)
print(X_train.columns[X_train.columns.duplicated(keep=False)].value_counts())


Duplicate columns:
Index(['Sex_F', 'Sex_M', 'ChestPainType_ASY', 'ChestPainType_ATA',
       'ChestPainType_NAP', 'ChestPainType_TA', 'RestingECG_LVH',
       'RestingECG_Normal', 'RestingECG_ST', 'ExerciseAngina_N',
       'ExerciseAngina_Y', 'ST_Slope_Down', 'ST_Slope_Flat', 'ST_Slope_Up'],
      dtype='str')
Sex_F                2
Sex_M                2
ChestPainType_ASY    2
ChestPainType_ATA    2
ChestPainType_NAP    2
ChestPainType_TA     2
RestingECG_LVH       2
RestingECG_Normal    2
RestingECG_ST        2
ExerciseAngina_N     2
ExerciseAngina_Y     2
ST_Slope_Down        2
ST_Slope_Flat        2
ST_Slope_Up          2
Name: count, dtype: int64


In [ ]:
X_train = X_train.loc[:, ~X_train.columns.duplicated()]
X_test = X_test.loc[:, ~X_test.columns.duplicated()]
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("Duplicate columns:")
print(X_train.columns[X_train.columns.duplicated()])

X_train shape: (734, 20)
X_test shape: (184, 20)
Duplicate columns:
Index([], dtype='str')


In [19]:
logistic_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(random_state=42))
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

results = cross_validate(
    logistic_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

for metric in scoring:
    scores = results[f"test_{metric}"]
    print(f"{metric}: {scores.mean():.4f} (+/- {scores.std():.4f})")

accuracy: 0.8474 (+/- 0.0299)
precision: 0.8616 (+/- 0.0457)
recall: 0.8671 (+/- 0.0222)
f1: 0.8633 (+/- 0.0226)
roc_auc: 0.9218 (+/- 0.0346)


In [20]:
dt_pipeline = Pipeline([
    ("model", DecisionTreeClassifier(random_state=42))
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

results = cross_validate(
    dt_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

for metric in scoring:
    scores = results[f"test_{metric}"]
    print(f"{metric}: {scores.mean():.4f} (+/- {scores.std():.4f})")

accuracy: 0.8011 (+/- 0.0483)
precision: 0.8208 (+/- 0.0438)
recall: 0.8202 (+/- 0.0683)
f1: 0.8194 (+/- 0.0466)
roc_auc: 0.7988 (+/- 0.0473)


In [21]:
rf_pipeline = Pipeline([
    ("model", RandomForestClassifier(n_estimators=100, random_state=42))
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

results = cross_validate(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

for metric in scoring:
    scores = results[f"test_{metric}"]
    print(f"{metric}: {scores.mean():.4f} (+/- {scores.std():.4f})")

accuracy: 0.8488 (+/- 0.0305)
precision: 0.8501 (+/- 0.0435)
recall: 0.8868 (+/- 0.0403)
f1: 0.8667 (+/- 0.0244)
roc_auc: 0.9258 (+/- 0.0265)


In [22]:
knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier(n_neighbors=5))
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

results = cross_validate(
    knn_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

for metric in scoring:
    scores = results[f"test_{metric}"]
    print(f"{metric}: {scores.mean():.4f} (+/- {scores.std():.4f})")

accuracy: 0.8284 (+/- 0.0279)
precision: 0.8406 (+/- 0.0405)
recall: 0.8547 (+/- 0.0271)
f1: 0.8467 (+/- 0.0213)
roc_auc: 0.8952 (+/- 0.0314)


In [23]:
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(kernel="rbf", probability=True, random_state=42))
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

results = cross_validate(
    svm_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

for metric in scoring:
    scores = results[f"test_{metric}"]
    print(f"{metric}: {scores.mean():.4f} (+/- {scores.std():.4f})")

c:\Users\cocoh\Downloads\heart_failure_prediction\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\cocoh\Downloads\heart_failure_prediction\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\cocoh\Downloads\heart_failure_prediction\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\cocoh\Downloads\heart_failure_prediction\.venv\Lib\site-packages\sklearn\

accuracy: 0.8502 (+/- 0.0440)
precision: 0.8468 (+/- 0.0551)
recall: 0.8966 (+/- 0.0400)
f1: 0.8694 (+/- 0.0340)
roc_auc: 0.9118 (+/- 0.0327)


c:\Users\cocoh\Downloads\heart_failure_prediction\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
